# Tech Challenge - Fase 2
## Notebook 00 - Setup do Ambiente AWS S3 + Databricks + Unity Catalog

Objetivo: configurar o ambiente do projeto usando **AWS S3** como Data Lake, acessado pelo Databricks por meio de **Unity Catalog External Volume**.

Premissas:
- Bucket S3 disponível: `s3://s3tc2/`
- Storage Credential criada: `trioprd`
- External Location criada para o bucket S3 AWS
- External Volume usado pelo notebook: `workspace.default.vol_trio_drive`
- Nenhuma credencial sensível deve ser salva no código ou no `config.json`


In [0]:
%sql
-- 1. Criar/validar External Volume
-- Este comando só precisa ser executado uma vez.
-- Se o volume já existir, nada será alterado.

CREATE EXTERNAL VOLUME IF NOT EXISTS workspace.default.vol_trio_drive
LOCATION 's3://s3tc2/';


In [0]:
%sql
-- 2. Validar volumes disponíveis no schema workspace.default

SHOW VOLUMES IN workspace.default;


In [0]:
%python
# 3. Imports e configuração principal

from datetime import datetime
import json

PROJECT_NAME = "fiap_alfabetizacao"

# Caminho governado pelo Unity Catalog External Volume.
# Tudo que for escrito nesse caminho será armazenado fisicamente no S3 AWS: s3://s3tc2/
BASE_PATH = "/Volumes/workspace/default/vol_trio_drive"

RAW_PATH = f"{BASE_PATH}/raw"
BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
GOLD_PATH = f"{BASE_PATH}/gold"
STREAMING_PATH = f"{BASE_PATH}/streaming"
LOG_PATH = f"{BASE_PATH}/logs"
DOCS_PATH = f"{BASE_PATH}/docs"
CONFIG_PATH = f"{BASE_PATH}/config"

EXECUTION_DATE = datetime.now().strftime("%Y-%m-%d")

print("PROJECT_NAME:", PROJECT_NAME)
print("BASE_PATH:", BASE_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)


In [0]:
%python
# 4. Criar estrutura lógica de diretórios no S3 via External Volume

paths = [
    RAW_PATH,

    f"{BRONZE_PATH}/meta_brasil",
    f"{BRONZE_PATH}/meta_uf",
    f"{BRONZE_PATH}/alfabetizacao_municipio",
    f"{BRONZE_PATH}/alfabetizacao_uf",
    f"{BRONZE_PATH}/ts_aluno",

    f"{SILVER_PATH}/meta_brasil",
    f"{SILVER_PATH}/meta_uf",
    f"{SILVER_PATH}/alfabetizacao_municipio",
    f"{SILVER_PATH}/alfabetizacao_uf",
    f"{SILVER_PATH}/aluno",

    f"{GOLD_PATH}/alfabetizacao_uf",
    f"{GOLD_PATH}/alfabetizacao_municipio",
    f"{GOLD_PATH}/comparativo_meta_resultado_uf",
    f"{GOLD_PATH}/ranking_uf",
    f"{GOLD_PATH}/ranking_municipio",
    f"{GOLD_PATH}/base_modelo_ia",
    f"{GOLD_PATH}/matriz_risco_educacional",
    f"{GOLD_PATH}/kpi_executivo",
    f"{GOLD_PATH}/ia_resultados",
    f"{GOLD_PATH}/feature_importance",
    f"{GOLD_PATH}/metricas_modelo_ia",
    f"{GOLD_PATH}/exports_powerbi",

    f"{STREAMING_PATH}/input/indicadores_municipais",
    f"{STREAMING_PATH}/checkpoint/bronze_indicadores_municipais",
    f"{STREAMING_PATH}/checkpoint/silver_indicadores_municipais",
    f"{STREAMING_PATH}/checkpoint/gold_indicadores_incrementais",
    f"{STREAMING_PATH}/bronze_streaming/indicadores_municipais",
    f"{STREAMING_PATH}/silver_streaming/indicadores_municipais",
    f"{STREAMING_PATH}/gold_streaming/indicadores_incrementais",

    f"{LOG_PATH}/pipeline_execution",
    f"{LOG_PATH}/data_quality",
    f"{LOG_PATH}/streaming_metrics",
    f"{LOG_PATH}/finops",
    f"{LOG_PATH}/rejected/aluno",

    DOCS_PATH,
    CONFIG_PATH
]

for path in paths:
    dbutils.fs.mkdirs(path)

print("Estrutura criada/validada com sucesso no S3 via Unity Catalog Volume.")


In [0]:
%python
# 5. Definir arquivos de entrada esperados

input_files = {
    "meta_brasil": {
        "file_name": "BR_INE~1.CSV",
        "path": f"{RAW_PATH}/BR_INE~1.CSV",
        "sep": ","
    },
    "meta_uf": {
        "file_name": "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv",
        "path": f"{RAW_PATH}/br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv",
        "sep": ","
    },
    "alfabetizacao_municipio": {
        "file_name": "br_inep_avaliacao_alfabetizacao_municipio.csv",
        "path": f"{RAW_PATH}/br_inep_avaliacao_alfabetizacao_municipio.csv",
        "sep": ","
    },
    "alfabetizacao_uf": {
        "file_name": "br_inep_avaliacao_alfabetizacao_uf.csv",
        "path": f"{RAW_PATH}/br_inep_avaliacao_alfabetizacao_uf.csv",
        "sep": ","
    },
    "ts_aluno": {
        "file_name": "TS_ALUNO.csv",
        "path": f"{RAW_PATH}/TS_ALUNO.csv",
        "sep": ";"
    }
}

input_files


In [0]:
%python
# 6. Opcional: mover arquivos carregados na raiz do volume para raw/
# Use esta célula quando os CSVs forem enviados para:
# /Volumes/workspace/default/vol_trio_drive/
# em vez de:
# /Volumes/workspace/default/vol_trio_drive/raw/

root_files = [f.name for f in dbutils.fs.ls(BASE_PATH) if not f.isDir()]
expected_files = [cfg["file_name"] for cfg in input_files.values()]

for file_name in expected_files:
    if file_name in root_files:
        source_path = f"{BASE_PATH}/{file_name}"
        target_path = f"{RAW_PATH}/{file_name}"

        print(f"Movendo arquivo para raw/: {source_path} -> {target_path}")
        dbutils.fs.mv(source_path, target_path)

print("Verificação de arquivos na raiz concluída.")


In [0]:
%python
# 7. Salvar configuração única para os próximos notebooks

config = {
    "project_name": PROJECT_NAME,
    "base_path": BASE_PATH,
    "raw_path": RAW_PATH,
    "bronze_path": BRONZE_PATH,
    "silver_path": SILVER_PATH,
    "gold_path": GOLD_PATH,
    "streaming_path": STREAMING_PATH,
    "log_path": LOG_PATH,
    "docs_path": DOCS_PATH,
    "config_path": CONFIG_PATH,
    "execution_date": EXECUTION_DATE,
    "bucket_s3": "s3://s3tc2/",
    "storage_credential": "trioprd",
    "external_volume": "workspace.default.vol_trio_drive",
    "input_files": input_files
}

config_file_path = f"{CONFIG_PATH}/config.json"

dbutils.fs.put(
    config_file_path,
    json.dumps(config, indent=4, ensure_ascii=False),
    overwrite=True
)

print("Configuração salva em:", config_file_path)


In [0]:
%python
# 8. Validar leitura da configuração

config_loaded = json.loads(dbutils.fs.head(config_file_path))
config_loaded


In [0]:
%python
# 9. Validar estrutura criada no S3

for label, path in {
    "BASE_PATH": BASE_PATH,
    "RAW_PATH": RAW_PATH,
    "BRONZE_PATH": BRONZE_PATH,
    "SILVER_PATH": SILVER_PATH,
    "GOLD_PATH": GOLD_PATH,
    "STREAMING_PATH": STREAMING_PATH,
    "LOG_PATH": LOG_PATH,
    "DOCS_PATH": DOCS_PATH,
    "CONFIG_PATH": CONFIG_PATH
}.items():
    print("=" * 80)
    print(label, "->", path)
    display(dbutils.fs.ls(path))


In [0]:
%python
# 10. Checklist dos arquivos esperados na raw/

try:
    available_files = [file.name for file in dbutils.fs.ls(RAW_PATH)]
except Exception as e:
    available_files = []
    print("Erro ao listar RAW_PATH:", e)

for file_name in expected_files:
    if file_name in available_files:
        print(f"OK: {file_name}")
    else:
        print(f"PENDENTE: {file_name}")


In [0]:
%python
# 11. Função de inspeção rápida sem count obrigatório
# Evitamos count() por padrão para reduzir custo em arquivos grandes.

def preview_csv(file_path, sep=",", rows=5, infer_schema=True, do_count=False):
    reader = (
        spark.read
        .option("header", "true")
        .option("sep", sep)
        .option("encoding", "UTF-8")
    )

    if infer_schema:
        reader = reader.option("inferSchema", "true")

    df = reader.csv(file_path)

    print(f"Arquivo: {file_path}")
    print(f"Colunas: {len(df.columns)}")

    if do_count:
        print(f"Linhas: {df.count()}")

    df.printSchema()
    display(df.limit(rows))

    return df


In [0]:
%python
# 12. Validar leitura dos arquivos pequenos

for dataset, cfg in input_files.items():
    if dataset != "ts_aluno":
        print("=" * 80)
        print(f"Validando dataset: {dataset}")

        try:
            preview_csv(
                file_path=cfg["path"],
                sep=cfg["sep"],
                rows=5,
                infer_schema=True,
                do_count=False
            )
        except Exception as e:
            print(f"Erro ao ler {dataset}: {e}")


In [0]:
%python
# 13. Validar leitura do TS_ALUNO sem count()

try:
    df_aluno_preview = preview_csv(
        file_path=input_files["ts_aluno"]["path"],
        sep=input_files["ts_aluno"]["sep"],
        rows=5,
        infer_schema=True,
        do_count=False
    )
except Exception as e:
    print("Erro ao ler TS_ALUNO.")
    print(e)


## Observação de produção

A pasta `raw/` é a landing zone do projeto. Ela pode receber arquivos por:

1. Upload manual controlado para o bucket S3;
2. Upload via Databricks UI para o External Volume;
3. Pipeline externo de ingestão;

Neste projeto, os arquivos são armazenados fisicamente em:

```text
s3://s3tc2/raw/
```

e acessados no Databricks por:

```text
/Volumes/workspace/default/vol_trio_drive/raw/
```

Essa abordagem evita credenciais no código e utiliza governança via Unity Catalog.
